<a href="https://colab.research.google.com/github/Magwict/Hands-on-Large-Language-Model-Code-/blob/main/Chapter_5_%E4%B8%BB%E9%A2%98%E5%BB%BA%E6%A8%A1%E5%B9%B6%E5%8F%AF%E8%A7%86%E5%8C%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install bertopic datasets openai datamapplot

In [ ]:
!pip install --upgrade datasets

In [ ]:
#从hugging face下载数据集
from datasets import load_dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

#提取元数据
abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/617 [00:00<?, ?B/s]

data.csv:   0%|          | 0.00/53.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from sentence_transformers import SentenceTransformer

#为每一个摘要创造嵌入
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(abstracts,show_progress_bar=True)

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1405 [00:00<?, ?it/s]

In [ ]:
from umap import UMAP

#将输入嵌入从384维降低到2维,便于后续可视化
umap_model = UMAP(
    n_components=2,min_dist=0.0,metric='cosine',random_state=42
)

reduced_embeddings = umap_model.fit_transform(embeddings)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
from hdbscan import HDBSCAN

#拟合模型并提取聚类
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom'
).fit(reduced_embeddings)

clusters = hdbscan_model.labels_ #返回每个样本的簇标签，格式为整数数组

#打印聚类结果，簇的数量
len(set(clusters))

/usr/local/lib/python3.12/dist-packages/hdbscan/plots.py:448: SyntaxWarning: invalid escape sequence '\l'
  axis.set_ylabel('$\lambda$ value')
/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


155

In [ ]:
from bertopic import BERTopic

#用之前定义的模型训练新模型：使用 BERTopic 库训练一个主题建模模型
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, #降维模型，可以用PCA替换UMAP
    hdbscan_model=hdbscan_model, #聚类模型，可以用k-means替换hdbscan
    verbose=True
).fit(abstracts,embeddings) #输入摘要和高维向量

2025-09-24 18:06:56,385 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-24 18:07:37,301 - BERTopic - Dimensionality - Completed ✓
2025-09-24 18:07:37,303 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-24 18:07:38,983 - BERTopic - Cluster - Completed ✓
2025-09-24 18:07:38,996 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-09-24 18:07:42,876 - BERTopic - Representation - Completed ✓


In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,12858,-1_of_the_and_to,"[of, the, and, to, in, we, that, for, on, lang...",[ With the proliferation of its applications ...
1,0,2545,0_speech_asr_recognition_end,"[speech, asr, recognition, end, acoustic, spok...",[ Recent advances in text-to-speech (TTS) led...
2,1,1441,1_medical_clinical_biomedical_patient,"[medical, clinical, biomedical, patient, notes...",[ Recent years have seen particular interest ...
3,2,1252,2_dialogue_response_dialog_responses,"[dialogue, response, dialog, responses, conver...",[ While neural conversation models have shown...
4,3,948,3_summarization_summaries_summary_abstractive,"[summarization, summaries, summary, abstractiv...",[ Sentence summarization shortens given texts...
...,...,...,...,...,...
150,149,51,149_table_tables_tabular_reasoning,"[table, tables, tabular, reasoning, column, ll...","[ Table reasoning, which aims to generate the..."
151,150,51,150_cqa_question_answer_community,"[cqa, question, answer, community, questions, ...",[ Community Question Answering (CQA) is a wel...
152,151,50,151_arabic_sentiment_analysis_dialect,"[arabic, sentiment, analysis, dialect, sa, msa...","[ Social media such as Twitter, Facebook, etc..."
153,152,50,152_oie_openie_extraction_ie,"[oie, openie, extraction, ie, open, extraction...",[ Open Information Extraction (OIE) is the ta...


In [ ]:
topic_model.get_topic(0)

[('speech', np.float64(0.026446167075355062)),
 ('asr', np.float64(0.0175609577099786)),
 ('recognition', np.float64(0.012625284787080655)),
 ('end', np.float64(0.009497813473863692)),
 ('acoustic', np.float64(0.008789331289606784)),
 ('spoken', np.float64(0.0068389241751667966)),
 ('audio', np.float64(0.0066426001073515316)),
 ('speaker', np.float64(0.006349509043228593)),
 ('the', np.float64(0.006330059252104944)),
 ('model', np.float64(0.006275455502393749))]

# Representation Models

In [ ]:
#保存原始模型（没有使用生成模型），以便后续对比分析
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
#可视化主题词差异
def topic_differences(model,original_topics,nr_topics=5):
  """Show the differences in topic representations between two models"""
  df = pd.DataFrame(columns=["Topic","Original","Updated"])
  for topic in range(nr_topics):

    #提取两个模型中每个簇的前五个词
    og_words = "|".join(list(zip(*original_topics[topic]))[0][:5])
    new_words = "|".join(list(zip(*model.get_topic(topic)))[0][:5])
    df.loc[len(df)] = [topic,og_words,new_words]

  return df

In [ ]:
import pandas as pd
from bertopic.representation import KeyBERTInspired

#使用KeyBERTInspired更新主题表示，优化BERTopic的表示
representation_model = KeyBERTInspired()

#topic_model是之前用BERTopic库训练的主题建模模型
topic_model.update_topics(abstracts,representation_model=representation_model)

#展示主题差异
topic_differences(topic_model,original_topics)

,Topic,Original,Updated
0,0,speech|asr|recognition|end|acoustic,phonetic|speech|transcription|language|spoken
1,1,medical|clinical|biomedical|patient|notes,nlp|clinical|ehr|ehrs|annotated
2,2,dialogue|response|dialog|responses|conversation,dialogues|dialogue|conversational|conversation...
3,3,summarization|summaries|summary|abstractive|do...,summarization|summarizers|summaries|summary|ab...
4,4,translation|nmt|machine|neural|bleu,translation|translating|translate|translations...


In [ ]:
from bertopic.representation import MaximalMarginalRelevance

# Update our topic representations to MaximalMarginalRelevance
representation_model = MaximalMarginalRelevance(diversity=0.5)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

,Topic,Original,Updated
0,0,speech|asr|recognition|end|acoustic,asr|audio|speaker|model|slu
1,1,medical|clinical|biomedical|patient|notes,clinical|biomedical|patient|healthcare|extraction
2,2,dialogue|response|dialog|responses|conversation,response|systems|conversational|generation|dia...
3,3,summarization|summaries|summary|abstractive|do...,summarization|document|extractive|rouge|factual
4,4,translation|nmt|machine|neural|bleu,translation|nmt|neural|bleu|parallel


In [ ]:
#from transformers import pipeline
#from bertopic.representation import TextGeneration

#prompt = """I have a topic that contains the following documents:
#[DOCUMENTS]

#The topic is described by the following keywords: '[KEYWORDS]'.

#Based on the documents and keywords, what is this topic about?"""

# Update our topic representations using Flan-T5
#generator = pipeline("text2text-generation", model="google/flan-t5-small")
#representation_model = TextGeneration(
#    generator, prompt=prompt, doc_length=50, tokenizer="whitespace"
#)
#topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
#topic_differences(topic_model, original_topics)

In [ ]:
import openai
from bertopic.representation import OpenAI

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following
format:
topic: <short topic label>
"""
# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOURKEY", base_url="https://api.deepseek.com")
representation_model = OpenAI(
  client, model="deepseek-chat", exponential_backoff=True, chat=True,
prompt=prompt
)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

100%|██████████| 155/155 [09:29<00:00,  3.67s/it]


,Topic,Original,Updated
0,0,speech|asr|recognition|end|acoustic,Advances in Automatic Speech Recognition and S...
1,1,medical|clinical|biomedical|patient|notes,Clinical Note Information Extraction in Health...
2,2,dialogue|response|dialog|responses|conversation,Dialogue System Optimization and Learning Methods
3,3,summarization|summaries|summary|abstractive|do...,Unsupervised and Abstractive Text Summarizatio...
4,4,translation|nmt|machine|neural|bleu,Enhancing Neural Machine Translation with Data...


In [ ]:
#获取每个文档（每篇论文）所属的簇cluster
doc_topics = topic_model.topics_

# 使用GPT生成的主题标签替换数字标签
topic_labels = topic_model.generate_topic_labels()  # 获取生成的主题标签

In [ ]:
reduced_embeddings.shape

(44949, 2)

In [ ]:
# 获取主题信息
topic_info = topic_model.get_topic_info()

# 查看生成的主题标签
print(topic_info[['Topic', 'Name']])

     Topic                                               Name
0       -1  -1_Cross-modal transfer learning and neural ne...
1        0  0_Advances in Automatic Speech Recognition and...
2        1  1_Clinical Note Information Extraction in Heal...
3        2  2_Dialogue System Optimization and Learning Me...
4        3  3_Unsupervised and Abstractive Text Summarizat...
..     ...                                                ...
150    149     149_Table Reasoning with Large Language Models
151    150  150_Community Question Answering (CQA) Researc...
152    151      151_Arabic Sentiment Analysis on Social Media
153    152  152_Open Information Extraction (OIE) systems ...
154    153         153_Table Question Answering and Retrieval

[155 rows x 2 columns]


In [ ]:
# 获取主题信息并创建映射（截断为15个字符）
topic_label_mapping = {}
for _, row in topic_info.iterrows():
    topic_id = row['Topic']
    generated_label = row['Name']
    # 截断为前15个字符，如果超过15个字符添加省略号
    truncated_label = generated_label[:25] + "..." if len(generated_label) > 25 else generated_label
    topic_label_mapping[topic_id] = truncated_label

In [ ]:
print("主题标签映射：")
print(topic_label_mapping)

主题标签映射：
{-1: '-1_Cross-modal transfer l...', 0: '0_Advances in Automatic S...', 1: '1_Clinical Note Informati...', 2: '2_Dialogue System Optimiz...', 3: '3_Unsupervised and Abstra...', 4: '4_Enhancing Neural Machin...', 5: '5_Unsupervised Question-A...', 6: '6_Gender Bias Mitigation ...', 7: '7_Cross-lingual Transfer ...', 8: '8_Advances in Named Entit...', 9: '9_Advances in Relation Ex...', 10: '10_Multimodal Vision-Lang...', 11: '11_dependency parsing and...', 12: '12_Word Meta-Embeddings a...', 13: '13_Advances in Natural La...', 14: '14_Legal AI and Language ...', 15: '15_Advances in Text Class...', 16: '16_Compositional Distribu...', 17: '17_Machine Translation Qu...', 18: '18_Human and Model Explan...', 19: "19_Zipf's linguistic laws...", 20: '20_Interpretable Multi-Ho...', 21: '21_Scientific Literature ...', 22: '22_Knowledge Graph Comple...', 23: '23_Automated hate speech ...', 24: '24_Parameter-Efficient Fi...', 25: '25_controlled text genera...', 26: '26_Aspect-Based Sentimen

In [ ]:
# 获取主题频率，找到前40个最大的簇
topic_info = topic_model.get_topic_info()
top_40_topics = topic_info[topic_info.Topic != -1].head(40)['Topic'].tolist()

print("前40个簇的编号:", top_40_topics)

前40个簇的编号: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]


In [ ]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,12858,-1_Cross-modal transfer learning and neural ne...,[Cross-modal transfer learning and neural netw...,[ With the proliferation of its applications ...
1,0,2545,0_Advances in Automatic Speech Recognition and...,[Advances in Automatic Speech Recognition and ...,[ Recent advances in text-to-speech (TTS) led...
2,1,1441,1_Clinical Note Information Extraction in Heal...,[Clinical Note Information Extraction in Healt...,[ Recent years have seen particular interest ...
3,2,1252,2_Dialogue System Optimization and Learning Me...,[Dialogue System Optimization and Learning Met...,[ While neural conversation models have shown...
4,3,948,3_Unsupervised and Abstractive Text Summarizat...,[Unsupervised and Abstractive Text Summarizati...,[ Sentence summarization shortens given texts...
...,...,...,...,...,...
150,149,51,149_Table Reasoning with Large Language Models,[Table Reasoning with Large Language Models],"[ Table reasoning, which aims to generate the..."
151,150,51,150_Community Question Answering (CQA) Researc...,[Community Question Answering (CQA) Research a...,[ Community Question Answering (CQA) is a wel...
152,151,50,151_Arabic Sentiment Analysis on Social Media,[Arabic Sentiment Analysis on Social Media],"[ Social media such as Twitter, Facebook, etc..."
153,152,50,152_Open Information Extraction (OIE) systems ...,[Open Information Extraction (OIE) systems and...,[ Open Information Extraction (OIE) is the ta...


In [ ]:
# 过滤数据：只保留属于前20个簇的点和噪声点(-1)
indices_to_keep = []
filtered_labels = []

for i, topic_id in enumerate(topic_model.topics_):
    if topic_id in top_40_topics or topic_id == -1:
        indices_to_keep.append(i)
        filtered_labels.append(topic_id)

print(indices_to_keep) #文档的序列
print(filtered_labels) #文档对应的聚类编号

[2, 3, 7, 10, 11, 12, 13, 14, 16, 17, 19, 20, 21, 22, 23, 24, 26, 28, 30, 31, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 55, 58, 59, 60, 62, 63, 64, 65, 66, 67, 68, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 106, 108, 109, 111, 112, 113, 114, 115, 116, 118, 119, 120, 126, 127, 128, 130, 131, 132, 133, 134, 137, 139, 140, 141, 142, 144, 145, 150, 151, 157, 158, 159, 160, 161, 162, 163, 165, 166, 167, 170, 171, 173, 174, 175, 176, 179, 180, 181, 182, 184, 185, 186, 188, 189, 190, 191, 193, 195, 197, 200, 203, 204, 205, 206, 207, 210, 211, 217, 218, 220, 221, 222, 223, 224, 225, 226, 227, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 241, 242, 243, 245, 247, 248, 250, 251, 255, 256, 257, 258, 260, 261, 262, 263, 264, 265, 266, 267, 269, 271, 272, 273, 274, 275, 276, 277, 278, 279, 283, 285, 286, 287, 288, 289, 297, 298, 299, 300, 303, 304, 310, 312, 315, 316, 317, 318,

In [ ]:
# 过滤嵌入向量
filtered_embeddings = reduced_embeddings[indices_to_keep]

In [ ]:
# 直接使用映射字典替换
formatted_labels = []
for label in filtered_labels:
    if label == -1:
        formatted_labels.append("Unlabelled")
    else:
        # 使用大模型生成的标签
        formatted_labels.append(topic_label_mapping.get(label, f"Topic {label}"))

In [ ]:
print(indices_to_keep) #文档的序列
print(formatted_labels) #文档对应的聚类编号

[2, 3, 7, 10, 11, 12, 13, 14, 16, 17, 19, 20, 21, 22, 23, 24, 26, 28, 30, 31, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 55, 58, 59, 60, 62, 63, 64, 65, 66, 67, 68, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 106, 108, 109, 111, 112, 113, 114, 115, 116, 118, 119, 120, 126, 127, 128, 130, 131, 132, 133, 134, 137, 139, 140, 141, 142, 144, 145, 150, 151, 157, 158, 159, 160, 161, 162, 163, 165, 166, 167, 170, 171, 173, 174, 175, 176, 179, 180, 181, 182, 184, 185, 186, 188, 189, 190, 191, 193, 195, 197, 200, 203, 204, 205, 206, 207, 210, 211, 217, 218, 220, 221, 222, 223, 224, 225, 226, 227, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 241, 242, 243, 245, 247, 248, 250, 251, 255, 256, 257, 258, 260, 261, 262, 263, 264, 265, 266, 267, 269, 271, 272, 273, 274, 275, 276, 277, 278, 279, 283, 285, 286, 287, 288, 289, 297, 298, 299, 300, 303, 304, 310, 312, 315, 316, 317, 318,

In [ ]:
import datamapplot
import numpy as np
import matplotlib

matplotlib.rcParams["figure.dpi"] = 300

fig,ax = datamapplot.create_plot(
    filtered_embeddings,
    formatted_labels,
    figsize=(15,12),
    title="Top 40 Topics Visualization",
    )